# Shufflestorm Runner & Settings Configurator

This notebook clones the **Shufflestorm** project, allows you to configure its execution parameters in `settings.py`, runs the main distributed matching pipeline using PySpark, and displays the results.

### Objectives:
1. **Clone** the Shufflestorm repository.
2. **Configure** all parameters in `shufflestorm/config/settings.py`.
3. **Execute** the main pipeline (`main.py`) to run and evaluate matching strategies.
4. **Display** the performance results.

## 1. Clone the Repository

In [ ]:
import os
import shutil

repo_url = "https://github.com/Gatmatz/shufflestorm.git"
repo_dir = "shufflestorm_cloned"

# Clean up existing clone to ensure a fresh setup
if os.path.exists(repo_dir):
    print(f"Removing existing directory '{repo_dir}' to ensure fresh clone...")
    shutil.rmtree(repo_dir)

print(f"Cloning {repo_url} into '{repo_dir}'...")
!git clone {repo_url} {repo_dir}

# Change directory into the cloned repository
%cd {repo_dir}

## 2. Configure Settings

Modify the fields below to customize the run. This cell will automatically update the `shufflestorm/config/settings.py` file with your parameters.

Available datasets in the repository:
- `bikedekho-small` (Size: 500)
- `bikedekho` (Size: 4786)
- `bikewale` (Size: 9003)
- `citeseer` (Size: 200000)

In [ ]:
# ==========================================
# Configure your settings here:
# ==========================================

DATASET = "bikewale"        # Dataset name (e.g. "bikedekho-small", "bikedekho", "bikewale", "citeseer")
DATASET_SIZE = 9003         # Number of records in the dataset (500, 4786, 9003, 200000)
REDUCER_SIZE = 200          # For Group-Based Matcher (number of records per reducer)
P_PRIME = 3                 # For Afrati-Ullman Model (prime number for mapping)

# ==========================================
# Write the configuration file
# ==========================================
settings_content = f"""# Dataset Parameters

DATASET = \"{DATASET}\"
DATASET_SIZE = {DATASET_SIZE}

# For Group-Based Matcher
REDUCER_SIZE = {REDUCER_SIZE}

# For Afrati-Ullman Model
P_PRIME = {P_PRIME}
"""

config_path = os.path.join("shufflestorm", "config", "settings.py")

with open(config_path, "w") as f:
    f.write(settings_content)

print(f"Successfully wrote new configuration to {config_path}:")
print("-" * 40)
print(settings_content)
print("-" * 40)

## 3. Install Dependencies (if needed)

If PySpark or other required packages are not installed in your environment, you can install them using this cell.

In [ ]:
import sys

# Check PySpark installation and install if missing
try:
    import pyspark
    print("PySpark is already installed.")
except ImportError:
    print("PySpark not found. Installing dependencies...")
    !{sys.executable} -m pip install pyspark numpy pandas python-dotenv

## 4. Run the Shufflestorm Pipeline

Run `main.py` using the current python executable. This will run:
1. **Naïve Strategy**
2. **Group-based Strategy**
3. **Spark SQL & Optimizer**
4. **Afrati-Ullmann Strategy**

The execution metrics (Execution Time, Task Count, Shuffle Volume) will be printed below and saved to the `results/` directory as JSON.

In [ ]:
import sys

print("Running main.py ...")
!{sys.executable} main.py

## 5. Visualize and Compare Results

We can read the generated JSON results file and display it in a neat DataFrame comparison.

In [ ]:
import json
import os
import pandas as pd

results_file = f"results/{DATASET}_results.json"

if os.path.exists(results_file):
    with open(results_file, "r") as f:
        data = json.load(f)
    
    print("Configuration Used:")
    for k, v in data["settings"].items():
        print(f"  {k}: {v}")
    print("\nStrategy Performance Comparison:")
    
    df_results = pd.DataFrame(data["results"]).T
    # Format the columns for readability
    if "time" in df_results.columns:
        df_results["time"] = df_results["time"].map(lambda x: f"{x:.4f} s" if isinstance(x, (int, float)) else x)
    if "shuffle_volume_bytes" in df_results.columns:
        df_results["shuffle_volume_bytes"] = df_results["shuffle_volume_bytes"].map(lambda x: f"{x:,} bytes" if isinstance(x, (int, float)) else x)
    
    display(df_results)
else:
    print(f"Results file '{results_file}' not found. Make sure main.py ran successfully.")